In [3]:
# api_integration.py 
"""
api_integration.py 
Combines database query + API call.

- Reads ONE city from the SQLite database (customers table)
- Uses Open-Meteo geocoding + current weather 
- Prints a short result
"""

import sqlite3
import urllib.request
import urllib.error
import json


DB_PATH = "data/myproject.db"


def fetch_json(url: str):
    """Fetch JSON from a URL and return parsed object."""
    with urllib.request.urlopen(url) as response:
        return json.loads(response.read().decode("utf-8"))


def get_one_city(db_path: str) -> str | None:
    """Return one city from the customers table."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        row = cursor.execute("SELECT city FROM customers LIMIT 1;").fetchone()
    return row[0] if row else None


def main():
    print("=== Optional API Integration ===\n")

    # 1) DB query: get one city
    city = get_one_city(DB_PATH)
    if not city:
        print("No city found in database.")
        return

    print(f"City from DB: {city}")

    try:
        # 2) API call #1: geocoding (city -> lat/lon)
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
        geo_data = fetch_json(geo_url)

        if "results" not in geo_data or not geo_data["results"]:
            print("Geocoding failed: city not found.")
            return

        r = geo_data["results"][0]
        lat, lon = r["latitude"], r["longitude"]

        # 3) API call #2: current weather
        weather_url = (
            "https://api.open-meteo.com/v1/forecast"
            f"?latitude={lat}&longitude={lon}&current_weather=true"
        )
        weather_data = fetch_json(weather_url)
        current = weather_data.get("current_weather")

        if not current:
            print("Weather fetch failed.")
            return

        temp = current.get("temperature")
        wind = current.get("windspeed")

        print(f"Current weather in {city}: {temp}°C, wind {wind} km/h")
        print("\nOptional API integration complete!")

    except urllib.error.URLError as e:
        print(f"API Error: {e}")
    except (KeyError, json.JSONDecodeError) as e:
        print(f"Parsing Error: {e}")


if __name__ == "__main__":
    main()


=== Optional API Integration ===

City from DB: Boston
Current weather in Boston: 0.4°C, wind 7.7 km/h

Optional API integration complete!
